# Clasificador de triatominos — MobileNetV3 (4 clases)
Chipos **iguales que antes**. "Otro" = insectos comunes en Venezuela + animales/plantas + objetos, escenas y personas (datasets adjuntos).


## Instalar dependencias


In [ ]:
!pip install -q -U datasets

## Configuración


In [ ]:
import os, io, time, glob, gc, random, collections, requests
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# IMPORTANTE: guardar todo en /kaggle/working (si no, va a una carpeta temporal y se pierde).
os.makedirs("/kaggle/working", exist_ok=True); os.chdir("/kaggle/working")

CLASES  = ["Panstrongylus", "Rhodnius", "Triatoma", "Otro"]
GENEROS = CLASES[:3]
g2i = {g: i for i, g in enumerate(CLASES)}

TOPE_CHIPO = 1500

OTRO_BICHOS     = ["Blattodea", "Coleoptera", "Formicidae", "Araneae", "Orthoptera",
                   "Lepidoptera", "Hymenoptera", "Pentatomidae", "Coreidae", "Culicidae", "Diptera"]
OTRO_NATURALEZA = ["Mammalia", "Aves", "Plantae", "Fungi", "Reptilia"]
TOPE_BICHOS     = 350
TOPE_NATURALEZA = 800
TOPE_GENERICO   = 11000

print("GPU:", tf.config.list_physical_devices("GPU"))

## Descargar y cargar datos


In [ ]:
UA = {"User-Agent": "triatominos"}

# GET con reintentos y respeto del rate-limit (para que NO queden taxones en 0).
def _get(url, params, intentos=5):
    for k in range(intentos):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=30)
            if r.status_code == 429:
                time.sleep(10); continue
            return r.json()
        except Exception:
            time.sleep(4)
    return {}

def inat(taxon, n):
    urls, page, vacios = [], 1, 0
    while len(urls) < n and page <= 80:
        j = _get("https://api.inaturalist.org/v1/observations", {
            "taxon_name": taxon, "quality_grade": "research", "photos": "true",
            "per_page": 50, "page": page})
        res = j.get("results", [])
        if not res:
            vacios += 1
            if vacios >= 3:   # 3 páginas vacías seguidas -> de verdad no hay más
                break
            time.sleep(3); page += 1; continue
        vacios = 0
        for o in res:
            f = o.get("photos") or []
            if f: urls.append(f[0]["url"].replace("square", "medium"))
            if len(urls) >= n: break
        page += 1; time.sleep(1.2)
    return urls[:n]

def gbif(genero, n):
    j = _get("https://api.gbif.org/v1/species/match", {"name": genero})
    key = j.get("usageKey")
    urls, off = [], 0
    while len(urls) < n and off < 2000:
        j = _get("https://api.gbif.org/v1/occurrence/search",
                 {"taxonKey": key, "mediaType": "StillImage", "limit": 50, "offset": off})
        res = j.get("results", [])
        if not res: break
        for o in res:
            for m in o.get("media", []):
                if m.get("identifier"): urls.append(m["identifier"]); break
            if len(urls) >= n: break
        off += 50; time.sleep(0.6)
    return urls[:n]

def bajar(url):
    r = requests.get(url, headers=UA, timeout=30)
    return np.asarray(Image.open(io.BytesIO(r.content)).convert("RGB").resize((224, 224)), np.uint8)

def bajar_taxones(taxones, tope, etiqueta):
    for taxon in taxones:
        antes = len(yo)
        for u in inat(taxon, tope):
            try: Xo.append(bajar(u)); yo.append(g2i["Otro"])
            except Exception: pass
        print(etiqueta, taxon, len(yo) - antes)
        time.sleep(2)   # espaciar entre taxones para no chocar con el rate-limit


from datasets import load_dataset

# 1) CHIPOS: TODAS las del dataset aumentado (Miranda), igual que antes.
ds = load_dataset("Totan2305/triatominos-augmentado-parquet")["train"]
nombres = ds.features["label"].names
n = ds.num_rows
Xm = np.empty((n, 224, 224, 3), np.uint8)
ym = np.empty(n, np.int64)
for i, ej in enumerate(ds):
    Xm[i] = np.asarray(ej["image"].convert("RGB").resize((224, 224)), np.uint8)
    ym[i] = g2i[nombres[ej["label"]]]
print("chipos Miranda:", n)

# 2) CHIPOS extra: iNaturalist + GBIF (campo real).
Xc, yc = [], []
for g in GENEROS:
    for fn in (inat, gbif):
        for u in fn(g, TOPE_CHIPO):
            try: Xc.append(bajar(u)); yc.append(g2i[g])
            except Exception: pass
    print("chipo campo", g, sum(1 for v in yc if v == g2i[g]))

# 3) OTRO: insectos comunes en Venezuela + animales/plantas (con reintentos).
Xo, yo = [], []
bajar_taxones(OTRO_BICHOS, TOPE_BICHOS, "Otro/insecto")
bajar_taxones(OTRO_NATURALEZA, TOPE_NATURALEZA, "Otro/naturaleza")
print("Otro (bichos+naturaleza):", len(yo))

# 4) OTRO: objetos, escenas y PERSONAS de los datasets adjuntos (Add Data en Kaggle).
exts = ("jpg", "jpeg", "png", "JPEG", "JPG", "PNG")
rutas = []
for base in glob.glob("/kaggle/input/*"):
    for e in exts:
        rutas += glob.glob(os.path.join(base, "**", "*." + e), recursive=True)
random.shuffle(rutas)
print("imágenes cotidianas disponibles:", len(rutas))
ng = 0
for r in rutas:
    if ng >= TOPE_GENERICO: break
    try:
        Xo.append(np.asarray(Image.open(r).convert("RGB").resize((224, 224)), np.uint8))
        yo.append(g2i["Otro"]); ng += 1
    except Exception: pass
print("Otro/cotidiano cargadas:", ng)

# Unir todo (liberando memoria intermedia).
Xc = np.array(Xc, np.uint8); Xo = np.array(Xo, np.uint8)
X = np.concatenate([Xm, Xc, Xo]); del Xm, Xc, Xo; gc.collect()
y = np.concatenate([ym, np.array(yc), np.array(yo)])
print("TOTAL:", len(X), collections.Counter(CLASES[l] for l in y))

idx = np.arange(len(X))
idx_tr, idx_te = train_test_split(idx, test_size=0.15, stratify=y, random_state=SEED)

## Preparar datos y modelo


In [ ]:
AUTO = tf.data.AUTOTUNE

def make_ds(indices, shuffle=False):
    def gen():
        for i in indices:
            yield X[i], y[i]
    d = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((224, 224, 3), tf.uint8), tf.TensorSpec((), tf.int64)))
    if shuffle:
        d = d.shuffle(6000, seed=SEED)
    d = d.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTO)
    return d.batch(32).prefetch(AUTO)

def construir():
    base = keras.applications.MobileNetV3Large(
        input_shape=(224, 224, 3), include_top=False,
        weights="imagenet", include_preprocessing=True)
    base.trainable = False
    model = keras.Sequential([
        keras.Input((224, 224, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(len(CLASES), activation="softmax"),
    ])
    return model, base

## Entrenamiento


In [ ]:
train_ds = make_ds(idx_tr, shuffle=True)
test_ds  = make_ds(idx_te)
w = compute_class_weight("balanced", classes=np.arange(len(CLASES)), y=y[idx_tr])
pesos = dict(enumerate(w))

model, base = construir()
model.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=pesos)

base.trainable = True
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
model.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=pesos)

## Evaluación


In [ ]:
y_true = y[idx_te]
y_pred = model.predict(test_ds).argmax(1)
print(classification_report(y_true, y_pred, target_names=CLASES, digits=4))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(cmap="Blues", xticks_rotation=45)
plt.tight_layout(); plt.savefig("/kaggle/working/matriz_confusion.png", dpi=150); plt.show()

## Guardar el modelo (queda en /kaggle/working -> panel Output)


In [ ]:
model.save("/kaggle/working/cnn_4clases.keras")
print("Guardado en /kaggle/working:", os.listdir("/kaggle/working"))

## Convertir a TFLite int8 (para el teléfono)


In [ ]:
def rep_data():
    for i in idx_tr[:300]:
        yield [X[i:i+1].astype(np.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl = conv.convert()
open("/kaggle/working/triatominos_int8.tflite", "wb").write(tfl)

it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
print("CLASES (orden de salida):", CLASES)
print("input  scale/zero:", it.get_input_details()[0]["quantization"])
print("output scale/zero:", it.get_output_details()[0]["quantization"])
print("Archivos en /kaggle/working:", os.listdir("/kaggle/working"))
from IPython.display import FileLink
display(FileLink("/kaggle/working/triatominos_int8.tflite"))